# Импорты зависимостей

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, 
    GradientBoostingClassifier, StackingClassifier
)
from sklearn.inspection import permutation_importance
import json
import joblib
import os
from datetime import datetime

# Загрузка данных и первичный анализ

In [ ]:
df = pd.read_csv(r'data\S06-hw-dataset-01.csv')

print(f"Размер датасета: {df.shape}")

print("\nИнформация о типах данных:")
print(df.info())

print("\nСтатистики числовых признаков:")
print(df.describe())

print("\nПроверка пропусков:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])
if missing_values.sum() == 0:
    print("Пропусков нет.")
else:
    print("Имеются пропуски.")

### Определяем признаки и таргет

In [ ]:
target_counts = df['target'].value_counts()
target_percent = df['target'].value_counts(normalize=True) * 100

print(f"Распределение классов:")
print(f"Класс 0: {target_counts[0]} ({target_percent[0]:.2f}%)")
print(f"Класс 1: {target_counts[1]} ({target_percent[1]:.2f}%)")

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
bars = plt.bar(['Class 0', 'Class 1'], target_counts.values, color=['skyblue', 'salmon'])
plt.title('Распределение классов (абсолютное)')
plt.ylabel('Количество')
for bar, count in zip(bars, target_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, 
             str(count), ha='center', va='bottom')

plt.subplot(1, 2, 2)
plt.pie(target_counts.values, labels=['Class 0', 'Class 1'], 
        autopct='%1.1f%%', colors=['skyblue', 'salmon'])
plt.title('Распределение классов (относительное)')

plt.tight_layout()
plt.savefig('artifacts/figures/target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nДисбаланс классов: {target_percent[0]/target_percent[1]:.2f}:1")
if abs(target_percent[0] - target_percent[1]) > 20:
    print("Сильный дисбаланс классов.")
elif abs(target_percent[0] - target_percent[1]) > 10:
    print("Умеренный дисбаланс классов.")
else:
    print("Незначительный дисбаланс классов.")

### Анализ корреляций

In [ ]:
numeric_cols = [col for col in df.columns if col.startswith('num') or col == 'tenure_months']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False)
plt.title('Матрица корреляций числовых признаков')
plt.tight_layout()
plt.savefig('artifacts/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

target_corr = {}
for col in df.columns:
    if col not in ['id', 'target'] and df[col].dtype in ['int64', 'float64']:
        target_corr[col] = abs(df[col].corr(df['target']))

top_corr = pd.Series(target_corr).sort_values(ascending=False).head(10)
print("\nТоп-10 признаков по абсолютной корреляции с таргетом:")
print(top_corr)

plt.figure(figsize=(10, 6))
top_corr.sort_values().plot(kind='barh', color='steelblue')
plt.title('Топ-10 признаков по корреляции с целевой переменной')
plt.xlabel('Абсолютная корреляция')
plt.tight_layout()
plt.savefig('artifacts/figures/top_correlations.png', dpi=300, bbox_inches='tight')
plt.show()

# Train/Test-сплит и воспроизводимость

In [ ]:
X = df.drop(['id', 'target'], axis=1)
y = df['target']

print(f"Признаки (X): {X.shape}")
print(f"Целевая переменная (y): {y.shape}")

# Проверяем типы данных
print("\nТипы признаков:")
print(X.dtypes.value_counts())

# Кодируем категориальные признаки (они уже числовые, но нужно убедиться)
categorical_cols = ['cat_contract', 'cat_region', 'cat_payment']
for col in categorical_cols:
    print(f"\n{col}: {sorted(X[col].unique())} уникальных значений")

RANDOM_STATE = 42
TEST_SIZE = 0.25

print(f"\nРазделяем данные:")
print(f"Размер теста: {TEST_SIZE * 100}%")
print(f"Random state: {RANDOM_STATE}")
print(f"Стратификация: Да")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, 
    stratify=y, shuffle=True
)

print(f"\nРазмеры после разделения:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

# Проверяем распределение классов в train и test
print(f"\nРаспределение классов в train: {np.bincount(y_train) / len(y_train) * 100}")
print(f"Распределение классов в test: {np.bincount(y_test) / len(y_test) * 100}")

Почему стратификация важна:
1) Сохраняет распределение классов в train и test.
2) Обеспечивает репрезентативность выборок.
3) Особенно важно при дисбалансе классов.
Почему фиксированный random_state (seed) важен:
1) Обеспечивает воспроизводимость результатов.
2) Позволяет сравнивать разные модели на одинаковых данных.
3) Облегчает отладку и валидацию.

# Baseline’ы

In [ ]:
# Функция для оценки модели
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """Оценка модели с сохранением метрик"""
    
    # Обучение модели
    model.fit(X_train, y_train)
    
    # Предсказания
    y_pred = model.predict(X_test)
    y_pred_proba = None
    
    # Пробуем получить вероятности
    try:
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    except:
        pass
    
    # Метрики
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred, average='binary'),
    }
    
    # ROC-AUC если есть вероятности
    if y_pred_proba is not None:
        try:
            metrics['roc_auc'] = roc_auc_score(y_test, y_pred_proba)
        except:
            metrics['roc_auc'] = None
    
    # Матрица ошибок
    cm = confusion_matrix(y_test, y_pred)
    
    # Вывод результатов
    print(f"\n{model_name}:")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  F1-score:  {metrics['f1']:.4f}")
    if metrics.get('roc_auc') is not None:
        print(f"  ROC-AUC:   {metrics['roc_auc']:.4f}")
    
    print(f"\n  Матрица ошибок:")
    print(f"  {cm[0][0]:4d}  {cm[0][1]:4d}")
    print(f"  {cm[1][0]:4d}  {cm[1][1]:4d}")
    
    return metrics, cm, model


print("\n1. DUMMY CLASSIFIER (most_frequent)")
dummy_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy_metrics, dummy_cm, dummy_model = evaluate_model(
    dummy_clf, X_train, y_train, X_test, y_test, "DummyClassifier"
)



logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(
        random_state=RANDOM_STATE, 
        max_iter=1000,
        class_weight='balanced'
    ))
])

logreg_metrics, logreg_cm, logreg_model = evaluate_model(
    logreg_pipeline, X_train, y_train, X_test, y_test, "LogisticRegression"
)

DummyClassifier предсказывает самый частый класс (всегда 0). Accuracy = доля класса 0 = 0.6767. Это нижняя граница производительности - любая разумная модель должна быть лучше.

In [ ]:
print("\n2. LOGISTIC REGRESSION (с масштабированием)")

logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(
        random_state=RANDOM_STATE, 
        max_iter=1000,
        class_weight='balanced'  # учёт дисбаланса
    ))
])

logreg_metrics, logreg_cm, logreg_model = evaluate_model(
    logreg_pipeline, X_train, y_train, X_test, y_test, "LogisticRegression"
)


LogisticRegression - простейшая линейная модель. Class_weight='balanced' помогает с дисбалансом классов. Если эта модель показывает хорошие результаты, данные вероятно линейно разделимы. Если результаты плохие, возможно нужны более сложные модели (деревья, ансамбли).

# Модели недели 6

In [ ]:
# Функция для подбора гиперпараметров и оценки.
def train_and_tune_model(model, param_grid, X_train, y_train, X_test, y_test, model_name, cv=3):
    """Обучение с подбором гиперпараметров"""
    try:
        cv_stratified = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
        grid_search = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            cv=cv_stratified,
            scoring='roc_auc',
            n_jobs=1,
            verbose=1,
            error_score='raise'
        )
        
        print("Подбираем гиперпараметры на train с помощью GridSearchCV...")
        grid_search.fit(X_train, y_train)
        
        print(f"Лучшие параметры: {grid_search.best_params_}")
        print(f"Лучший ROC-AUC на CV: {grid_search.best_score_:.4f}")
        
        # Оценка лучшей модели на тесте.
        best_model = grid_search.best_estimator_
        
        # Вычисляем метрики.
        y_pred = best_model.predict(X_test)
        y_pred_proba = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, 'predict_proba') else None
        
        test_metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'f1': f1_score(y_test, y_pred, average='binary'),
            'roc_auc': roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
        }
        
        print(f"\nМетрики на тесте:")
        print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
        print(f"  F1-score: {test_metrics['f1']:.4f}")
        if test_metrics['roc_auc'] is not None:
            print(f"  ROC-AUC:  {test_metrics['roc_auc']:.4f}")
        
        return {
            'best_model': best_model,
            'best_params': grid_search.best_params_,
            'best_cv_score': grid_search.best_score_,
            'test_metrics': test_metrics,
            'cv_results': grid_search.cv_results_
        }
        
    except Exception as e:
        print(f"Ошибка при обучении модели {model_name}: {e}")
        print("Используем модель с дефолтными параметрами...")
        
        # Обучаем модель с дефолтными параметрами.
        model.fit(X_train, y_train)
        
        # Вычисляем метрики.
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
        
        test_metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'f1': f1_score(y_test, y_pred, average='binary'),
            'roc_auc': roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
        }
        
        return {
            'best_model': model,
            'best_params': model.get_params(),
            'best_cv_score': None,
            'test_metrics': test_metrics,
            'cv_results': None
        }


# Модель 1: DecisionTreeClassifier.
print("\n1. DECISION TREE CLASSIFIER")

# Определяем параметры для перебора.
tree_param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy'],
    'class_weight': ['balanced', None]
}

tree_clf = DecisionTreeClassifier(random_state=RANDOM_STATE)

tree_results = train_and_tune_model(
    tree_clf, tree_param_grid, X_train, y_train, X_test, y_test,
    "DecisionTreeClassifier", cv=5
)

# Визуализация лучшего дерева.
best_tree = tree_results['best_model']
plt.figure(figsize=(20, 10))
plot_tree(best_tree, 
          feature_names=X.columns,
          class_names=['Class 0', 'Class 1'],
          filled=True, 
          rounded=True,
          max_depth=3,
          fontsize=10)
plt.title('Лучшее дерево решений (первые 3 уровня)', fontsize=14)
plt.tight_layout()
plt.savefig('artifacts/figures/best_decision_tree.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nАнализ сложности дерева:")
print(f"Глубина дерева: {best_tree.get_depth()}")
print(f"Количество листьев: {best_tree.get_n_leaves()}")
print(f"Количество признаков: {X.shape[1]}")

# Важность признаков для дерева.
tree_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_tree.feature_importances_
}).sort_values('importance', ascending=False)

print("\nТоп-10 важных признаков по DecisionTree:")
print(tree_importance.head(10))

# Визуализация важности признаков
plt.figure(figsize=(12, 6))
plt.barh(tree_importance.head(15)['feature'][::-1], 
         tree_importance.head(15)['importance'][::-1],
         color='steelblue')
plt.title('Важность признаков (Decision Tree)', fontsize=14)
plt.xlabel('Важность')
plt.tight_layout()
plt.savefig('artifacts/figures/tree_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()


# Модель 2: RandomForestClassifier.
print("\n2. RANDOM FOREST CLASSIFIER (оптимизированная версия)")

print("Быстрая оценка с дефолтными параметрами...")
rf_quick = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=4,
    random_state=RANDOM_STATE,
    n_jobs=2
)

rf_quick.fit(X_train, y_train)
y_pred_proba = rf_quick.predict_proba(X_test)[:, 1]
roc_auc_quick = roc_auc_score(y_test, y_pred_proba)

print(f"ROC-AUC (быстрый): {roc_auc_quick:.4f}")

if len(X_train) < 10000:
    print("\nЗапускаем оптимизированный GridSearch...")
    
    rf_param_grid = {
        'n_estimators': [50, 100],
        'max_depth': [5, 10],
        'min_samples_leaf': [2, 4],
        'max_features': ['sqrt', 0.5]
    }
    
    cv_stratified = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    rf_clf = RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=2,  # Ограничиваем ядра
        class_weight='balanced'
    )
    
    grid_search = GridSearchCV(
        estimator=rf_clf,
        param_grid=rf_param_grid,
        cv=cv_stratified,
        scoring='roc_auc',
        n_jobs=1,
        verbose=1,
        error_score='raise'
    )
    
    print("Начинаем подбор параметров...")
    grid_search.fit(X_train, y_train)
    
    print(f"Лучшие параметры: {grid_search.best_params_}")
    print(f"Лучший ROC-AUC на CV: {grid_search.best_score_:.4f}")
    
    # Оценка лучшей модели.
    best_rf = grid_search.best_estimator_
    
    rf_metrics, rf_cm, _ = evaluate_model(
        best_rf, X_train, y_train, X_test, y_test, "RandomForestClassifier (optimized)"
    )
    
    rf_results = {
        'best_model': best_rf,
        'best_params': grid_search.best_params_,
        'best_cv_score': grid_search.best_score_,
        'test_metrics': rf_metrics,
        'cv_results': grid_search.cv_results_
    }
    
else:
    rf_results = {
        'best_model': rf_quick,
        'best_params': {'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 4},
        'best_cv_score': None,
        'test_metrics': {'accuracy': accuracy_score(y_test, rf_quick.predict(X_test)),
                        'f1': f1_score(y_test, rf_quick.predict(X_test)),
                        'roc_auc': roc_auc_quick}
    }

# Анализ Random Forest.
best_rf = rf_results['best_model']
print(f"\nАнализ Random Forest:")
print(f"Количество деревьев: {best_rf.n_estimators}")
print(f"Глубина деревьев: {best_rf.max_depth}")
print(f"Используемые признаки при разделении: {best_rf.max_features}")

# Важность признаков.
rf_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nТоп-10 важных признаков по RandomForest:")
print(rf_importance.head(10))

# Визуализация.
plt.figure(figsize=(12, 6))
plt.barh(rf_importance.head(15)['feature'][::-1], 
        rf_importance.head(15)['importance'][::-1],
        color='darkgreen')
plt.title('Важность признаков: Random Forest', fontsize=14)
plt.xlabel('Важность')
plt.tight_layout()
plt.savefig('artifacts/figures/rf_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()


# Модель 3: GradientBoostingClassifier.
print("\n3. GRADIENT BOOSTING CLASSIFIER")

# Быстрая оценка.
print("Быстрая оценка с дефолтными параметрами...")
gb_quick = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=RANDOM_STATE
)

gb_quick.fit(X_train, y_train)


gb_param_grid = {
    'n_estimators': [50, 100], 
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}

gb_clf = GradientBoostingClassifier(random_state=RANDOM_STATE)

cv_stratified = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=gb_clf,
    param_grid=gb_param_grid,
    cv=cv_stratified,
    scoring='roc_auc',
    n_jobs=1,
    verbose=1
)

print("Начинаем подбор параметров...")
grid_search.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший ROC-AUC на CV: {grid_search.best_score_:.4f}")

gb_results = {
    'best_model': grid_search.best_estimator_,
    'best_params': grid_search.best_params_,
    'best_cv_score': grid_search.best_score_,
    'test_metrics': None
}

# Анализ кривой обучения для Gradient Boosting.
best_gb = gb_results['best_model']

# Получаем кривую обучения.
train_scores = []
test_scores = []

for i, y_pred in enumerate(best_gb.staged_predict(X_train)):
    train_scores.append(f1_score(y_train, y_pred))
    
for i, y_pred in enumerate(best_gb.staged_predict(X_test)):
    test_scores.append(f1_score(y_test, y_pred))

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(train_scores) + 1), train_scores, label='Train', linewidth=2)
plt.plot(range(1, len(test_scores) + 1), test_scores, label='Test', linewidth=2)
plt.xlabel('Количество деревьев')
plt.ylabel('F1-score')
plt.title('Кривая обучения Gradient Boosting')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('artifacts/figures/gb_learning_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nАнализ Gradient Boosting:")
print(f"Learning rate: {best_gb.learning_rate}")
print(f"Количество деревьев: {best_gb.n_estimators}")
print(f"Глубина деревьев: {best_gb.max_depth}")


# Модель 4: StackingClassifier.
print("\n4. STACKING CLASSIFIER")

# Базовые модели для стекинга.
base_models = [
    ('dt', DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=4,
        random_state=RANDOM_STATE
    )),
    ('rf', RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )),
    ('gb', GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=RANDOM_STATE
    ))
]

# Мета-модель.
meta_model = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    class_weight='balanced'
)

# Создаем стекинг-классификатор.
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1,
    passthrough=False
)

print("StackingClassifier с 3 базовыми моделями:")
print("1. Decision Tree (упрощенный)")
print("2. Random Forest")
print("3. Gradient Boosting")
print("Мета-модель: LogisticRegression")

# Обучаем и оцениваем.
stacking_metrics, stacking_cm, stacking_model = evaluate_model(
    stacking_clf, X_train, y_train, X_test, y_test, "StackingClassifier"
)

stacking_results = {
    'best_model': stacking_model,
    'best_params': {'base_models': ['dt', 'rf', 'gb'], 'meta_model': 'LogisticRegression'},
    'best_cv_score': None,
    'test_metrics': stacking_metrics
}

# Сравнение всех моделей

In [ ]:
# Функция для вычисления метрик модели.
def compute_model_metrics(model, X_train, y_train, X_test, y_test):
    """Вычисляет метрики для модели"""
    try:
        if not hasattr(model, 'classes_'):
            model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        y_pred_proba = None
        
        try:
            y_pred_proba = model.predict_proba(X_test)[:, 1]
        except:
            y_pred_proba = None
        
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'f1': f1_score(y_test, y_pred, average='binary'),
            'roc_auc': None
        }
        
        if y_pred_proba is not None:
            try:
                metrics['roc_auc'] = roc_auc_score(y_test, y_pred_proba)
            except:
                metrics['roc_auc'] = None
        
        return metrics
        
    except Exception as e:
        print(f"Ошибка при вычислении метрик: {e}")
        return {'accuracy': None, 'f1': None, 'roc_auc': None}

# Собираем результаты всех моделей с гарантированными метриками.
all_results = {}

# 1. Dummy.
print("Вычисляем метрики для DummyClassifier...")
all_results['Dummy'] = {
    'metrics': compute_model_metrics(dummy_model, X_train, y_train, X_test, y_test),
    'model': dummy_model,
    'cv_score': None
}

# 2. LogisticRegression.
print("Вычисляем метрики для LogisticRegression...")
all_results['LogisticRegression'] = {
    'metrics': compute_model_metrics(logreg_model, X_train, y_train, X_test, y_test),
    'model': logreg_model,
    'cv_score': None
}

# 3. DecisionTree.
print("Вычисляем метрики для DecisionTree...")
if 'tree_results' in locals() and tree_results is not None:
    metrics = compute_model_metrics(tree_results['best_model'], X_train, y_train, X_test, y_test)
    all_results['DecisionTree'] = {
        'metrics': metrics,
        'model': tree_results['best_model'],
        'cv_score': tree_results.get('best_cv_score', None)
    }
else:
    print("DecisionTree результаты не найдены")
    all_results['DecisionTree'] = {
        'metrics': {'accuracy': None, 'f1': None, 'roc_auc': None},
        'model': None,
        'cv_score': None
    }

# 4. RandomForest.
print("Вычисляем метрики для RandomForest...")
if 'rf_results' in locals() and rf_results is not None:
    metrics = compute_model_metrics(rf_results['best_model'], X_train, y_train, X_test, y_test)
    all_results['RandomForest'] = {
        'metrics': metrics,
        'model': rf_results['best_model'],
        'cv_score': rf_results.get('best_cv_score', None)
    }
else:
    print("RandomForest результаты не найдены")
    all_results['RandomForest'] = {
        'metrics': {'accuracy': None, 'f1': None, 'roc_auc': None},
        'model': None,
        'cv_score': None
    }

# 5. GradientBoosting.
print("Вычисляем метрики для GradientBoosting...")
if 'gb_results' in locals() and gb_results is not None:
    metrics = compute_model_metrics(gb_results['best_model'], X_train, y_train, X_test, y_test)
    all_results['GradientBoosting'] = {
        'metrics': metrics,
        'model': gb_results['best_model'],
        'cv_score': gb_results.get('best_cv_score', None)
    }
else:
    print("GradientBoosting результаты не найдены")
    all_results['GradientBoosting'] = {
        'metrics': {'accuracy': None, 'f1': None, 'roc_auc': None},
        'model': None,
        'cv_score': None
    }

# 6. Stacking (если был создан).
if 'stacking_model' in locals():
    print("Вычисляем метрики для Stacking...")
    all_results['Stacking'] = {
        'metrics': compute_model_metrics(stacking_model, X_train, y_train, X_test, y_test),
        'model': stacking_model,
        'cv_score': None
    }
else:
    print("Stacking модель не найдена")


search_summaries = {}

if 'tree_results' in locals() and tree_results.get('best_params'):
    search_summaries['DecisionTree'] = {
        'best_params': tree_results['best_params'],
        'cv_score': float(tree_results['best_cv_score']) if tree_results['best_cv_score'] else None
    }

if 'rf_results' in locals() and rf_results.get('best_params'):
    search_summaries['RandomForest'] = {
        'best_params': rf_results['best_params'],
        'cv_score': float(rf_results['best_cv_score']) if rf_results['best_cv_score'] else None
    }

if 'gb_results' in locals() and gb_results.get('best_params'):
    search_summaries['GradientBoosting'] = {
        'best_params': gb_results['best_params'],
        'cv_score': float(gb_results['best_cv_score']) if gb_results['best_cv_score'] else None
    }


with open('artifacts/search_summaries.json', 'w') as f:
    json.dump(search_summaries, f, indent=2)

# Создаем DataFrame для сравнения.
comparison_data = []
for model_name, results in all_results.items():
    # Проверяем, что metrics существует и является словарем.
    if results['metrics'] is not None and isinstance(results['metrics'], dict):
        row = {
            'Model': model_name,
            'Accuracy': results['metrics'].get('accuracy', None),
            'F1-score': results['metrics'].get('f1', None),
            'ROC-AUC': results['metrics'].get('roc_auc', None)
        }
        if 'cv_score' in results and results['cv_score'] is not None:
            row['CV ROC-AUC'] = results['cv_score']
        comparison_data.append(row)
    else:
        print(f"Внимание: нет метрик для модели {model_name}")

if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    print("\nСравнение моделей (метрики на тесте):")
    print(comparison_df.to_string(index=False))
    
    # Визуализация сравнения.
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Accuracy (игнорируем None значения).
    acc_data = comparison_df[['Model', 'Accuracy']].dropna()
    if not acc_data.empty:
        axes[0, 0].barh(range(len(acc_data)), acc_data['Accuracy'].values, color='skyblue')
        axes[0, 0].set_yticks(range(len(acc_data)))
        axes[0, 0].set_yticklabels(acc_data['Model'])
        axes[0, 0].set_xlabel('Accuracy')
        axes[0, 0].set_title('Сравнение Accuracy')
        axes[0, 0].invert_yaxis()
    else:
        axes[0, 0].text(0.5, 0.5, 'Нет данных', ha='center', va='center')
        axes[0, 0].set_title('Сравнение Accuracy')
    
    # F1-score (игнорируем None значения).
    f1_data = comparison_df[['Model', 'F1-score']].dropna()
    if not f1_data.empty:
        axes[0, 1].barh(range(len(f1_data)), f1_data['F1-score'].values, color='lightgreen')
        axes[0, 1].set_yticks(range(len(f1_data)))
        axes[0, 1].set_yticklabels(f1_data['Model'])
        axes[0, 1].set_xlabel('F1-score')
        axes[0, 1].set_title('Сравнение F1-score')
        axes[0, 1].invert_yaxis()
    else:
        axes[0, 1].text(0.5, 0.5, 'Нет данных', ha='center', va='center')
        axes[0, 1].set_title('Сравнение F1-score')
    
    # ROC-AUC (игнорируем None значения).
    roc_data = comparison_df[['Model', 'ROC-AUC']].dropna()
    if not roc_data.empty:
        axes[1, 0].barh(range(len(roc_data)), roc_data['ROC-AUC'].values, color='salmon')
        axes[1, 0].set_yticks(range(len(roc_data)))
        axes[1, 0].set_yticklabels(roc_data['Model'])
        axes[1, 0].set_xlabel('ROC-AUC')
        axes[1, 0].set_title('Сравнение ROC-AUC')
        axes[1, 0].invert_yaxis()
        axes[1, 0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
    else:
        axes[1, 0].text(0.5, 0.5, 'Нет данных', ha='center', va='center')
        axes[1, 0].set_title('Сравнение ROC-AUC')
    
    # CV ROC-AUC (если есть).
    cv_data = comparison_df[['Model', 'CV ROC-AUC']].dropna() if 'CV ROC-AUC' in comparison_df.columns else pd.DataFrame()
    if not cv_data.empty:
        axes[1, 1].barh(range(len(cv_data)), cv_data['CV ROC-AUC'].values, color='gold')
        axes[1, 1].set_yticks(range(len(cv_data)))
        axes[1, 1].set_yticklabels(cv_data['Model'])
        axes[1, 1].set_xlabel('CV ROC-AUC')
        axes[1, 1].set_title('Сравнение CV ROC-AUC')
        axes[1, 1].invert_yaxis()
        axes[1, 1].axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
    else:
        axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.savefig('artifacts/figures/model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Определяем лучшую модель по ROC-AUC (игнорируем None).
    valid_roc = comparison_df[comparison_df['ROC-AUC'].notna()]
    if not valid_roc.empty:
        best_idx = valid_roc['ROC-AUC'].idxmax()
        best_model_name = valid_roc.loc[best_idx, 'Model']
        best_model_info = all_results.get(best_model_name, {})
        
        print(f"\n{'='*60}")
        print(f"ЛУЧШАЯ МОДЕЛЬ: {best_model_name}")
        print(f"{'='*60}")
        
        if best_model_info.get('metrics'):
            print(f"Accuracy:  {best_model_info['metrics'].get('accuracy', 'N/A'):.4f}")
            print(f"F1-score:  {best_model_info['metrics'].get('f1', 'N/A'):.4f}")
            print(f"ROC-AUC:   {best_model_info['metrics'].get('roc_auc', 'N/A'):.4f}")
        
        if 'cv_score' in best_model_info and best_model_info['cv_score'] is not None:
            print(f"CV ROC-AUC: {best_model_info['cv_score']:.4f}")
    else:
        print("\nНе удалось определить лучшую модель (нет ROC-AUC метрик)")
        
else:
    print("Нет данных для сравнения моделей")


# Интерпретация

In [ ]:
# Определяем лучшую модель по ROC-AUC.
best_model_name = comparison_df.loc[comparison_df['ROC-AUC'].idxmax(), 'Model']
best_model_info = all_results[best_model_name]
best_model = best_model_info['model']

print(f"Лучшая модель: {best_model_name}")
print(f"ROC-AUC: {best_model_info['metrics']['roc_auc']:.4f}")

# 1. Permutation Importance.

print("Вычисляем важность признаков через перемешивание...")
print(f"Количество повторений: 10")
print(f"Метрика для оценки: ROC-AUC")

perm_importance = permutation_importance(
    best_model, 
    X_test, 
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=2,
    scoring='roc_auc'
)

# Создаем DataFrame с результатами.
perm_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

print("\nТоп-15 признаков по Permutation Importance:")
print(perm_df.head(15).to_string(index=False))

# 2. Визуализация Permutation Importance.

plt.figure(figsize=(12, 8))
top_n = 15
features = perm_df.head(top_n)['feature'][::-1]
means = perm_df.head(top_n)['importance_mean'][::-1]
stds = perm_df.head(top_n)['importance_std'][::-1]

bars = plt.barh(range(top_n), means, xerr=stds, 
                color='steelblue', alpha=0.7, ecolor='black', capsize=5)
plt.yticks(range(top_n), features)
plt.xlabel('Уменьшение ROC-AUC при перемешивании признака', fontsize=12)
plt.title(f'Permutation Importance для {best_model_name}\n(топ-{top_n} признаков)', fontsize=14)
plt.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='Нулевая важность')

# Добавляем значения на график.
for i, (mean, std) in enumerate(zip(means, stds)):
    plt.text(mean + std + 0.001, i, f'{mean:.3f}', 
             va='center', fontsize=9)

plt.legend()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('artifacts/figures/permutation_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Детальный анализ топ-признаков.

top_features = perm_df.head(5)
for i, (idx, row) in enumerate(top_features.iterrows(), 1):
    feature = row['feature']
    importance = row['importance_mean']
    std = row['importance_std']
    
    print(f"\n{i}. {feature}")
    print(f"   Важность: {importance:.4f} ± {std:.4f}")
    
    # Анализ типа признака.
    if feature.startswith('num'):
        print(f"   Тип: Числовой (нормализованный)")
        print(f"   Статистики по датасету:")
        print(f"     - Среднее: {df[feature].mean():.3f}")
        print(f"     - Стандартное отклонение: {df[feature].std():.3f}")
        print(f"     - Минимум: {df[feature].min():.3f}")
        print(f"     - Максимум: {df[feature].max():.3f}")
        
        # Корреляция с таргетом.
        corr = df[feature].corr(df['target'])
        print(f"     - Корреляция с таргетом: {corr:.3f}")
        
        if abs(corr) > 0.3:
            print(f"     → Сильная корреляция с таргетом")
        elif abs(corr) > 0.1:
            print(f"     → Умеренная корреляция с таргетом")
        else:
            print(f"     → Слабая корреляция с таргетом")
            
    elif feature.startswith('cat'):
        print(f"   Тип: Категориальный")
        unique_vals = sorted(df[feature].unique())
        print(f"   Уникальные значения: {unique_vals}")
        
        # Распределение значений.
        value_counts = df[feature].value_counts().sort_index()
        print(f"   Распределение значений:")
        for val, count in value_counts.items():
            percentage = count / len(df) * 100
            print(f"     - {val}: {count} ({percentage:.1f}%)")
            
        # Связь с таргетом по категориям.
        print(f"   Целевое распределение по категориям:")
        for val in unique_vals:
            subset = df[df[feature] == val]
            if len(subset) > 0:
                target_mean = subset['target'].mean()
                print(f"     - {val}: средний таргет = {target_mean:.3f}")
                
    elif feature == 'tenure_months':
        print(f"   Тип: Длительность (месяцы)")
        print(f"   Статистики:")
        print(f"     - Минимум: {df[feature].min()} месяцев")
        print(f"     - Максимум: {df[feature].max()} месяцев")
        print(f"     - Среднее: {df[feature].mean():.1f} месяцев")
        print(f"     - Медиана: {df[feature].median()} месяцев")
        
        # Корреляция с таргетом.
        corr = df[feature].corr(df['target'])
        print(f"     - Корреляция с таргетом: {corr:.3f}")
        
        if corr > 0:
            print(f"     → Положительная связь: больше месяцев → выше вероятность класса 1")
        else:
            print(f"     → Отрицательная связь: больше месяцев → ниже вероятность класса 1")

# 4. Сравнение с ожиданиями.
print("\nАнализ результатов Permutation Importance:")

# Проверяем числовые признаки.
numeric_important = [f for f in top_features['feature'] if f.startswith('num')]
if numeric_important:
    print(f"1. Числовые признаки в топе: {len(numeric_important)} из 5")
    print(f"   Это ожидаемо, так как числовые признаки часто несут информацию")
    
    # Проверяем, есть ли признаки с высокой корреляцией.
    high_corr_features = []
    for feature in numeric_important:
        corr = abs(df[feature].corr(df['target']))
        if corr > 0.2:
            high_corr_features.append((feature, corr))
    
    if high_corr_features:
        print(f"   Признаки с высокой корреляцией с таргетом:")
        for feature, corr in high_corr_features:
            print(f"   - {feature}: корреляция = {corr:.3f}")
    else:
        print(f"   Удивительно: важные признаки имеют низкую линейную корреляцию")
        print(f"   Это может указывать на нелинейные зависимости")

# Проверяем категориальные признаки.
categorical_important = [f for f in top_features['feature'] if f.startswith('cat')]
if categorical_important:
    print(f"\n2. Категориальные признаки в топе: {len(categorical_important)} из 5")
    print(f"   Это важно, так как категориальные признаки могут кодировать")
    print(f"   качественную информацию (например, тип контракта, регион)")
    
    # Проверяем распределение категорий.
    for feature in categorical_important:
        print(f"\n   Анализ {feature}:")
        for category in sorted(df[feature].unique()):
            subset = df[df[feature] == category]
            target_rate = subset['target'].mean()
            print(f"   - Категория {category}: {len(subset)} samples, "
                  f"таргет rate = {target_rate:.3f}")

# Проверяем tenure_months.
if 'tenure_months' in top_features['feature'].values:
    print(f"\n3. Признак tenure_months в топе")
    print(f"   Ожидаемо: длительность часто важна для классификации")
    print(f"   Например: новые клиенты vs постоянные клиенты")
    
    high_tenure = df[df['tenure_months'] > df['tenure_months'].median()]
    low_tenure = df[df['tenure_months'] <= df['tenure_months'].median()]
    
    print(f"   Сравнение групп:")
    print(f"   - Высокая длительность (> {df['tenure_months'].median():.0f} мес): "
          f"таргет rate = {high_tenure['target'].mean():.3f}")
    print(f"   - Низкая длительность (≤ {df['tenure_months'].median():.0f} мес): "
          f"таргет rate = {low_tenure['target'].mean():.3f}")

# 5. Проверка на утечки данных.

suspicious_features = []
for feature in perm_df['feature']:
    if feature == 'target':
        suspicious_features.append(feature)
    elif perm_df.loc[perm_df['feature'] == feature, 'importance_mean'].iloc[0] > 0.5:
        suspicious_features.append(feature)

if suspicious_features:
    print(f"ВНИМАНИЕ: Обнаружены потенциально проблемные признаки:")
    for feature in suspicious_features:
        importance = perm_df.loc[perm_df['feature'] == feature, 'importance_mean'].iloc[0]
        print(f"  - {feature}: важность = {importance:.3f}")
    print("\n  Рекомендации:")
    print("  1. Проверить, нет ли утечек данных")
    print("  2. Убедиться, что признаки не содержат информацию о таргете")
    print("  3. Рассмотреть удаление этих признаков")
else:
    print("Потенциальных утечек данных не обнаружено.")

# 6. Сравнение с Feature Importances модели.

if hasattr(best_model, 'feature_importances_'):
    print(f"Модель {best_model_name} имеет встроенные feature importances")
    
    # Получаем feature importances.
    model_importances = pd.DataFrame({
        'feature': X.columns,
        'model_importance': best_model.feature_importances_
    }).sort_values('model_importance', ascending=False)
    
    # Объединяем с permutation importances.
    comparison = pd.merge(
        perm_df[['feature', 'importance_mean']],
        model_importances[['feature', 'model_importance']],
        on='feature'
    )
    
    # Нормализуем для сравнения.
    comparison['perm_norm'] = comparison['importance_mean'] / comparison['importance_mean'].max()
    comparison['model_norm'] = comparison['model_importance'] / comparison['model_importance'].max()
    
    print("\nСравнение топ-10 признаков:")
    top_comparison = comparison.head(10)
    print(top_comparison[['feature', 'perm_norm', 'model_norm']].to_string(index=False))
    
    # Визуализация сравнения.
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Permutation Importance.
    axes[0].barh(top_comparison['feature'][::-1], top_comparison['perm_norm'][::-1], 
                color='steelblue', alpha=0.7)
    axes[0].set_xlabel('Нормализованная важность')
    axes[0].set_title('Permutation Importance (топ-10)')
    axes[0].invert_yaxis()
    
    # Model Feature Importance.
    axes[1].barh(top_comparison['feature'][::-1], top_comparison['model_norm'][::-1],
                color='darkgreen', alpha=0.7)
    axes[1].set_xlabel('Нормализованная важность')
    axes[1].set_title(f'{best_model_name} Feature Importance (топ-10)')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('artifacts/figures/importance_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Анализ различий.
    print("\nАнализ различий между методами:")
    
    # Находим признаки, которые сильно отличаются.
    comparison['diff'] = abs(comparison['perm_norm'] - comparison['model_norm'])
    different_features = comparison[comparison['diff'] > 0.3].head(3)
    
    if not different_features.empty:
        print("Признаки с наибольшими различиями в важности:")
        for _, row in different_features.iterrows():
            print(f"  - {row['feature']}:")
            print(f"    Permutation: {row['perm_norm']:.3f}, "
                  f"Model: {row['model_norm']:.3f}, "
                  f"Разница: {row['diff']:.3f}")
        print("\n  Возможные причины различий:")
        print("  1. Признак взаимодействует с другими признаками")
        print("  2. Модель переоценивает/недооценивает важность")
        print("  3. Permutation importance более надежен для коррелированных признаков")
    else:
        print("Методы дают согласованные результаты - хороший признак!")
        
else:
    print(f"Модель {best_model_name} не имеет встроенных feature importances")

# 7. Выводы и рекомендации.

print("\nКлючевые выводы из анализа важности признаков:")

# Резюме по топ-признакам.
print(f"1. Наиболее важные признаки для {best_model_name}:")
for i, (idx, row) in enumerate(top_features.iterrows(), 1):
    print(f"   {i}. {row['feature']} (важность: {row['importance_mean']:.3f})")

perm_results = {
    'best_model': best_model_name,
    'permutation_importance': perm_df.to_dict('records'),
    'top_features': top_features.to_dict('records'),
    'interpretation': {
        'numeric_in_top': len(numeric_important),
        'categorical_in_top': len(categorical_important),
        'has_tenure': 'tenure_months' in top_features['feature'].values,
        'suspicious_features': suspicious_features
    }
}

metrics_test_data = {}
for model_name, results in all_results.items():
    if results['metrics'] is not None:
        metrics_test_data[model_name] = {
            'accuracy': float(results['metrics']['accuracy']) if results['metrics']['accuracy'] is not None else None,
            'f1': float(results['metrics']['f1']) if results['metrics']['f1'] is not None else None,
            'roc_auc': float(results['metrics']['roc_auc']) if results['metrics']['roc_auc'] is not None else None,
        }

with open('artifacts/metrics_test.json', 'w') as f:
    json.dump(metrics_test_data, f, indent=2)

# Сохранение лучшей модели

In [ ]:
joblib.dump(best_model, 'artifacts/best_model.joblib')

# Сохраняем метаданные
best_model_meta = {
    'best_model_name': best_model_name,
    'test_metrics': {
        'accuracy': float(best_model_info['metrics']['accuracy']) if best_model_info['metrics'].get('accuracy') is not None else None,
        'f1': float(best_model_info['metrics']['f1']) if best_model_info['metrics'].get('f1') is not None else None,
        'roc_auc': float(best_model_info['metrics']['roc_auc']) if best_model_info['metrics'].get('roc_auc') is not None else None,
    },
    'cv_score': float(best_model_info.get('cv_score')) if best_model_info.get('cv_score') is not None else None,
    'timestamp': datetime.now().isoformat()
}

with open('artifacts/best_model_meta.json', 'w') as f:
    json.dump(best_model_meta, f, indent=2)